# Adaptador de datos — población perinatal NCHS

Etapa del framework: adaptador de datos (E2). Para cada año de parto 2016–2023 une, en una fila por producto: los nacidos vivos de la cohorte vinculada con su muerte de 0 a 6 días (`libraries/nchs_reader.py`) y las muertes fetales del archivo público de muertes fetales (`libraries/perinatal_builder.py`). Solo conserva los campos prenatales presentes en los dos archivos (ficha del desenlace, opción A). Lee los ZIP de `../NCHS_data/raw/` y `../NCHS_data/raw/fetal/` sin descomprimirlos y escribe `data/interim/perinatal_<año>.parquet` (no se versiona). Versión en inglés: `01_build_cohorts.ipynb` (mismo código).

Requisitos: `pandas`, `numpy` y `pyarrow`. Cada año tarda alrededor de 1 minuto y usa unos 3 GB de memoria.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from libraries.perinatal_builder import build_perinatal, YEARS, SHARED
from libraries.nchs_reader import OUT_DIR

REBUILD = False      # True: rebuild every year even if its Parquet file exists
OUT_FILE = ROOT / "data" / "outputs_01_build_perinatal.txt"

## 1. Construir la población por año (se omiten los años que ya existen)

In [ ]:
for y in YEARS:
    if REBUILD or not (OUT_DIR / f"perinatal_{y}.parquet").exists():
        build_perinatal(y)          # about 1 minute and 3 GB of memory per year
    else:
        print(f"year {y}: already built")

## 2. Conteos agregados

Población: madres residentes en EE. UU., productos de 22 a 47 semanas. Incluye el porcentaje de valores desconocidos de cada campo prenatal por componente. Salida: `data/outputs_01_build_perinatal.txt`.

In [ ]:
# Codes "unknown or not stated" of each shared prenatal field (user guides)
UNKNOWN = {"MRACEHISP": [8], "MEDUC": [9], "MBSTATE_REC": [3], "PRIORLIVE": [99], "PRIORDEAD": [99],
           "ILLB_R": [999], "PRECARE": [99], "RF_CESARN": [99], "CIG_0": [99],
           "MHTR": [99], "CIG_1": [99], "DLMP_MM": [99], "FAGECOMB": [99]}
FIELDS = [c for c in SHARED if c not in ("RESTATUS", "DPLURAL", "MAGER")]
cols = ["component", "in_population", "label_main", "label_fetal", "label_early_neonatal", "ga_weeks"] + FIELDS

year_rows, unk_sum, n_comp = [], {}, {}
for y in YEARS:                      # one year at a time, to keep memory low
    p = pd.read_parquet(OUT_DIR / f"perinatal_{y}.parquet", columns=cols)
    p = p[p["in_population"]]
    ge28 = p["ga_weeks"] >= 28
    year_rows.append({"year": y, "products": len(p),
                      "live_births": int((p["component"] == "live_birth").sum()),
                      "fetal_deaths": int(p["label_fetal"].sum()),
                      "early_neonatal_deaths": int(p["label_early_neonatal"].sum()),
                      "perinatal_events": int(p["label_main"].sum()),
                      "products_ge28": int(ge28.sum()), "events_ge28": int((ge28 & p["label_main"]).sum())})
    for comp in ("live_birth", "fetal_death"):
        q = p[p["component"] == comp]
        n_comp[comp] = n_comp.get(comp, 0) + len(q)
        for c in FIELDS:
            s = q[c]
            if c == "BMI":
                unk = s.ge(99) | s.isna()       # 99.9 = unknown (float32, compared by threshold)
            else:
                unk = (s.isin(UNKNOWN[c]) if c in UNKNOWN else s.astype("string").eq("U")) | s.isna()
            unk_sum[(c, comp)] = unk_sum.get((c, comp), 0) + int(unk.fillna(True).sum())
    del p

by_year = pd.DataFrame(year_rows).set_index("year")
by_year["per_1000"] = (1000 * by_year["perinatal_events"] / by_year["products"]).round(2)
unknown = pd.DataFrame({
    "pct_unknown_live_births": {c: round(100 * unk_sum[(c, "live_birth")] / n_comp["live_birth"], 1) for c in FIELDS},
    "pct_unknown_fetal_deaths": {c: round(100 * unk_sum[(c, "fetal_death")] / n_comp["fetal_death"], 1) for c in FIELDS}})

lines = ["Perinatal population (live births and fetal deaths of 22-47 weeks, US-resident mothers), "
         "aggregate counts only", "", by_year.to_string(), "",
         "Percentage unknown or not stated by component (all years)", unknown.to_string()]
OUT_FILE.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))